此文件描述來源擴充前的整合快照。輸出／讀取位置已改為 snapshots/pre_source_expansion_20261019，以保留當時bytes，避免重跑舊版覆寫目前current。最新資料由 expand_candidate_sources_20261019.ipynb 產生。

# Data Description
- Author: RUEI-HSIANG, CHANG (B12303030)，Codex 協助
- Date first created: 20261019
- Background: 將已完成的身分審核、歷史出生／性別、競選人物頁與社群入口接入可直接閱讀的當前資料清單，避免各批證據分散且主表仍顯示未查找。
- Unit of observation: council background＝候選人；pending history＝候選人×可能對應的2022候選紀錄；current master＝候選人；current fields＝候選人×欄位。
- Sample period: 2026登記名單，歷史欄位來源年2022、當日整合2026-10-19。
- Known issues: 仍785位沒有MOI或本輪競選人物頁；其中157位找到同名同縣市2022線索，尚未核對身分，未填入已支持欄位。新人物頁25人不是全部背景完整。未認證學位、戶籍或2026現任；社群只整理入口，貼文窗口與貼文蒐集暫緩。
- Definition for each variable:

  - candidate_key、candidate_name、county_city、office_type、electoral_district、party_registration_raw：2026登記識別、姓名、縣市、公職、選區及推薦政黨原文。
  - birth_date_supported_history、birth_year_supported_history、sex_supported_history：先前身分審核有支持者的2022官方申報生日（民國轉西元）、出生年、性別；不是2026戶籍重新認定。空白保留未知。
  - birth_sex_evidence_status、history_identity_review：歷史來源與身分支持限制。history_source_file／row：原始資料檔與1起算列號。history_match_count：2022同名同縣市候選紀錄數，不能直接當作身分已核實。
  - education_raw、experience_raw、background_source_url：新競選頁或既有MOI頁原文及來源；education_2022_declared另存歷史申報層級。education_completion_status提醒未認證學位，不能把在學當完成。
  - moi_profile_url、campaign_profile_url：既有MOI頁、新競選人物頁；只有URL不代表四項背景均完整。
  - first_time_this_office_supported／first_time_any_office_supported：可支持終生非首次者填0；其餘留空，不以未命中填1。incumbency_2026留空，incumbency_status不以2022當選推定目前仍在任。
  - pending history表的history_*：2022原始姓名、縣市、選區、生日、性別、學歷及檔／列／ZIP成員；birth_date_history_candidate僅為可能對應者日期。source_sha256為原始檔hash；same_district_number只比較編號，不證明選區邊界相同；profile_evidence_present為有無兩批人物頁。
  - current主表／欄位表：沿用round4變數定義；本輪更新birth_date、sex、first_time_*及Facebook／Instagram／Threads來源狀態。existing_value為來源值、evidence為出處，status／note／next_source為支持程度與待辦，不可把status當模型屬性。
  - updated_date為本次整合日期。其餘原登記、資格、納入規則與旗標保持上游；未取得的貼文／追蹤數仍空白。

- Data path / Role / SHA-256:
  - data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv / background or collection inventory / 2b63c4a399d39788c02d8ae09010ae6bdeaccda6842f41191a1eeffe4c89f81e (1502列)
  - data/processed/council_history_pending_identity_20261019.csv / background or collection inventory / e88d1bca609d0a837dfc597e95d10d85a1dec12a5feaf8d542c6fc25388c143f (164列)
  - data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv / background or collection inventory / fbb32bba0c26e5b266fe4c7710058f27353f86fdbc8d59f7c112d4a30451f37d (1583列)
  - data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv / background or collection inventory / 6a9ac7fa74b768cf05a9ea58140f36a1567a2e62195c5a241cc0d0ce39e21e21 (31660列)
- Source data: round4主表／欄位表、council_identity_education_review_20261019.csv、council_campaign_profiles_20261019.csv、candidate_background_evidence_round3_20261019.csv、candidate_social_account_evidence_20261019.csv、candidate_profile_leads_20261019.csv、中選會2022六份原始elcand與六份elbase。逐檔hash見整合程式PINS。
- Processing code: data/code/integrate_current_inventory_20261019.ipynb。
- Role: background evidence / collection inventory；不是已決定研究樣本或分類模型輸入。
- Verification status: 以下以保存檔重新讀取，核對hash、列數、鍵及690人逐欄一致；整合程式逐人確認對應原始列姓名／生日／性別與審核表一致。
- Downstream analysis input: council_background_20261019.csv可提供有來源年代與支持狀態的背景欄位；缺失、學位完成、納入規則仍須依研究設計處理。current兩表為接續蒐集入口；舊日期／round版本保留追溯。


In [1]:
from pathlib import Path
import json,hashlib,pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
FRAMES={}


## Change for Your Own Data

In [2]:
assert sha(ROOT/'data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv')=='2b63c4a399d39788c02d8ae09010ae6bdeaccda6842f41191a1eeffe4c89f81e'
FRAMES['data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv']=pd.read_csv(ROOT/'data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv',dtype=str).fillna('')
assert len(FRAMES['data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv'])==1502
assert sha(ROOT/'data/processed/council_history_pending_identity_20261019.csv')=='e88d1bca609d0a837dfc597e95d10d85a1dec12a5feaf8d542c6fc25388c143f'
FRAMES['data/processed/council_history_pending_identity_20261019.csv']=pd.read_csv(ROOT/'data/processed/council_history_pending_identity_20261019.csv',dtype=str).fillna('')
assert len(FRAMES['data/processed/council_history_pending_identity_20261019.csv'])==164
assert sha(ROOT/'data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv')=='fbb32bba0c26e5b266fe4c7710058f27353f86fdbc8d59f7c112d4a30451f37d'
FRAMES['data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv']=pd.read_csv(ROOT/'data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv',dtype=str).fillna('')
assert len(FRAMES['data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv'])==1583
assert sha(ROOT/'data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv')=='6a9ac7fa74b768cf05a9ea58140f36a1567a2e62195c5a241cc0d0ce39e21e21'
FRAMES['data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv']=pd.read_csv(ROOT/'data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv',dtype=str).fillna('')
assert len(FRAMES['data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv'])==31660


## Summary

In [3]:
for p,d in FRAMES.items():print(p,{'rows':len(d),'columns':len(d.columns)})

data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv {'rows': 1502, 'columns': 26}
data/processed/council_history_pending_identity_20261019.csv {'rows': 164, 'columns': 18}
data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv {'rows': 1583, 'columns': 34}
data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv {'rows': 31660, 'columns': 11}


### Sample

In [4]:
for p,d in FRAMES.items():
 print(p);display(d.head(10))

data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv


,candidate_key,candidate_name,county_city,office_type,electoral_district,party_registration_raw,birth_date_supported_history,birth_year_supported_history,sex_supported_history,birth_sex_evidence_status,...,background_source_url,moi_profile_url,campaign_profile_url,education_2022_declared,education_completion_status,first_time_this_office_supported,first_time_any_office_supported,incumbency_2026,incumbency_status,updated_date
0,2026|直轄市議員|臺北市第1選舉區|謝閔弘,謝閔弘,臺北市,直轄市議員,臺北市第1選舉區,時代力量,,,,尚無可連結的歷史紀錄,...,https://www.taiwangogo.tw/hsieh-min-hung/,,https://www.taiwangogo.tw/hsieh-min-hung/,,保留來源原文，未認證學位；在學不當作已畢業,,0,,未由歷史當選或履歷自動推定,2026-10-19
1,2026|直轄市議員|臺北市第1選舉區|李文,李文,臺北市,直轄市議員,臺北市第1選舉區,無,,,,尚無可連結的歷史紀錄,...,,,,,保留來源原文，未認證學位；在學不當作已畢業,,,,未由歷史當選或履歷自動推定,2026-10-19
2,2026|直轄市議員|臺北市第1選舉區|陳彥廷,陳彥廷,臺北市,直轄市議員,臺北市第1選舉區,無,,,,尚無可連結的歷史紀錄,...,,,,,保留來源原文，未認證學位；在學不當作已畢業,,,,未由歷史當選或履歷自動推定,2026-10-19
3,2026|直轄市議員|臺北市第1選舉區|鍾佩玲,鍾佩玲,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1979-03-27,1979,女,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,,碩士,保留來源原文，未認證學位；在學不當作已畢業,0,0,,未由歷史當選或履歷自動推定,2026-10-19
4,2026|直轄市議員|臺北市第1選舉區|紀建漢,紀建漢,臺北市,直轄市議員,臺北市第1選舉區,無,,,,尚無可連結的歷史紀錄,...,,,,,保留來源原文，未認證學位；在學不當作已畢業,,,,未由歷史當選或履歷自動推定,2026-10-19
5,2026|直轄市議員|臺北市第1選舉區|陳賢蔚,陳賢蔚,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1977-07-12,1977,男,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,,大學,保留來源原文，未認證學位；在學不當作已畢業,0,0,,未由歷史當選或履歷自動推定,2026-10-19
6,2026|直轄市議員|臺北市第1選舉區|林世宗,林世宗,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1959-06-13,1959,男,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,,碩士,保留來源原文，未認證學位；在學不當作已畢業,0,0,,未由歷史當選或履歷自動推定,2026-10-19
7,2026|直轄市議員|臺北市第1選舉區|黃瀞瑩,黃瀞瑩,臺北市,直轄市議員,臺北市第1選舉區,台灣民眾黨,1992-06-10,1992,女,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,,碩士,保留來源原文，未認證學位；在學不當作已畢業,0,0,,未由歷史當選或履歷自動推定,2026-10-19
8,2026|直轄市議員|臺北市第1選舉區|林揚庭,林揚庭,臺北市,直轄市議員,臺北市第1選舉區,中國國民黨,,,,尚無可連結的歷史紀錄,...,,,,,保留來源原文，未認證學位；在學不當作已畢業,,,,未由歷史當選或履歷自動推定,2026-10-19
9,2026|直轄市議員|臺北市第1選舉區|陳慈慧,陳慈慧,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1983-11-22,1983,女,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,https://www.moi.gov.tw/LocalOfficial_Content.a...,https://www.moi.gov.tw/LocalOfficial_Content.a...,,大學,保留來源原文，未認證學位；在學不當作已畢業,0,0,,未由歷史當選或履歷自動推定,2026-10-19


data/processed/council_history_pending_identity_20261019.csv


,candidate_key,candidate_name,county_city,electoral_district,same_district_number,birth_date_history_candidate,identity_status,profile_evidence_present,history_name,history_county,history_district,history_birth_roc,history_sex,history_education,history_file,history_row,history_archive_member,source_sha256
0,2026|直轄市議員|臺北市第2選舉區|吳欣岱,吳欣岱,臺北市,臺北市第2選舉區,True,1987-07-01,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,True,吳欣岱,臺北市,2,0760701,女,大學,data/raw/cec_all_offices/107_elcand.csv,34,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
1,2026|直轄市議員|臺北市第3選舉區|林柏勛,林柏勛,臺北市,臺北市第3選舉區,True,1983-11-15,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,林柏勛,臺北市,3,0721115,男,碩士,data/raw/cec_all_offices/107_elcand.csv,43,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
2,2026|直轄市議員|臺北市第5選舉區|吳怡萱,吳怡萱,臺北市,臺北市第5選舉區,True,1989-08-17,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,吳怡萱,臺北市,5,0780817,女,大學,data/raw/cec_all_offices/107_elcand.csv,75,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
3,2026|直轄市議員|臺北市第6選舉區|陳聖文,陳聖文,臺北市,臺北市第6選舉區,True,1998-05-06,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,陳聖文,臺北市,6,0870506,男,碩士,data/raw/cec_all_offices/107_elcand.csv,112,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
4,2026|直轄市議員|新北市第2選舉區|張昭隆,張昭隆,新北市,新北市第2選舉區,False,1966-10-22,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,張昭隆,新北市,8,0551022,男,大學,data/raw/cec_all_offices/107_elcand.csv,318,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
5,2026|直轄市議員|新北市第4選舉區|董俊良,董俊良,新北市,新北市第4選舉區,True,1970-07-30,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,董俊良,新北市,4,0590730,男,高中(職),data/raw/cec_all_offices/107_elcand.csv,276,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
6,2026|直轄市議員|新北市第4選舉區|陳宛毓,陳宛毓,新北市,新北市第4選舉區,True,1988-04-18,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,True,陳宛毓,新北市,4,0770418,女,大學,data/raw/cec_all_offices/107_elcand.csv,266,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
7,2026|直轄市議員|新北市第5選舉區|戴瑋姗,戴瑋姗,新北市,新北市第5選舉區,True,1990-01-23,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,戴瑋姗,新北市,5,0790123,女,大學,data/raw/cec_all_offices/107_elcand.csv,282,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
8,2026|直轄市議員|新北市第6選舉區|張志豪,張志豪,新北市,新北市第6選舉區,True,1981-08-24,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,張志豪,新北市,6,0700824,男,碩士,data/raw/cec_all_offices/107_elcand.csv,307,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
9,2026|直轄市議員|新北市第7選舉區|羅文崇,羅文崇,新北市,新北市第7選舉區,True,1964-07-08,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,羅文崇,新北市,7,0530708,男,碩士,data/raw/cec_all_offices/107_elcand.csv,312,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...


data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv


,candidate_key,candidate_name,office_type,county_city,electoral_district,party_registration_raw,registration_source,registration_source_file,registration_source_page,qualification_inherited,...,Threads_status,posts_Facebook_status,posts_Instagram_status,posts_Threads_status,followers_status,policy_text_status,visual_status,political_family_status,administrative_experience_status,qualification_status
0,2026-C1-001,郭璽,直轄市長,臺北市,,台灣麻將最大黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
1,2026-C1-002,沈伯洋,直轄市長,臺北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,已保存來源連出的帳號入口；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
2,2026-C1-003,蕭文乾,直轄市長,臺北市,,臺灣SoR無法黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
3,2026-C1-004,唐新民,直轄市長,臺北市,,三勢團結促進聯盟,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
4,2026-C1-005,蔣萬安,直轄市長,臺北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
5,2026-C1-006,林志成,直轄市長,臺北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
6,2026-C1-007,蘇巧慧,直轄市長,新北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,已保存來源連出的帳號入口；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
7,2026-C1-008,李四川,直轄市長,新北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
8,2026-C1-009,蘇輝湟,直轄市長,新北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
9,2026-C1-010,張善政,直轄市長,桃園市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對


data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv


,candidate_key,candidate_name,office_type,county_city,field,status,existing_value,evidence,note,next_source,updated_date
0,2026-C1-001,郭璽,直轄市長,臺北市,sex,已有資料，待來源複核,男,https://db.cec.gov.tw/,dob_corroborated_by_media,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
1,2026-C1-001,郭璽,直轄市長,臺北市,birth_date,已有資料，待來源複核,1958-08-31,https://db.cec.gov.tw/,CEC_name_and_birth_date_corroborated_by_CNA,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
2,2026-C1-001,郭璽,直轄市長,臺北市,education,已有資料，待來源複核,大學,,歷史公報或媒體補充，非2026學歷認證,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
3,2026-C1-001,郭璽,直轄市長,臺北市,incumbency,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
4,2026-C1-001,郭璽,直轄市長,臺北市,first_time_this_office,既有搜尋後仍缺，待補查,,,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
5,2026-C1-001,郭璽,直轄市長,臺北市,first_time_any_office,已有資料，待來源複核,0,https://db.cec.gov.tw/,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
6,2026-C1-001,郭璽,直轄市長,臺北市,profile,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
7,2026-C1-001,郭璽,直轄市長,臺北市,website,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
8,2026-C1-001,郭璽,直轄市長,臺北市,Facebook,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19
9,2026-C1-001,郭璽,直轄市長,臺北市,Instagram,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19


### Summary Stats

In [5]:
c,h,m,f=FRAMES.values();display(c[['birth_date_supported_history','sex_supported_history','education_raw','experience_raw','first_time_any_office_supported']].ne('').sum().to_frame('非空白人數：支持程度依欄位說明'));display(h[['candidate_name','county_city','birth_date_history_candidate','same_district_number','profile_evidence_present']].head(20))

,非空白人數：支持程度依欄位說明
birth_date_supported_history,690
sex_supported_history,690
education_raw,640
experience_raw,628
first_time_any_office_supported,694


,candidate_name,county_city,birth_date_history_candidate,same_district_number,profile_evidence_present
0,吳欣岱,臺北市,1987-07-01,True,True
1,林柏勛,臺北市,1983-11-15,True,False
2,吳怡萱,臺北市,1989-08-17,True,False
3,陳聖文,臺北市,1998-05-06,True,False
4,張昭隆,新北市,1966-10-22,False,False
5,董俊良,新北市,1970-07-30,True,False
6,陳宛毓,新北市,1988-04-18,True,True
7,戴瑋姗,新北市,1990-01-23,True,False
8,張志豪,新北市,1981-08-24,True,False
9,羅文崇,新北市,1964-07-08,True,False


### Assertions

In [6]:
c,h,m,f=FRAMES.values()
assert len(c)==1502 and c.candidate_key.is_unique
assert len(m)==1583 and m.candidate_key.is_unique
assert len(f)==31660 and not f.duplicated(['candidate_key','field']).any()
assert c.birth_date_supported_history.ne('').sum()==690
assert c.sex_supported_history.isin(['男','女']).sum()==690
assert c.incumbency_2026.eq('').all()
assert set(c.first_time_any_office_supported)<= {'','0'}
assert set(c.first_time_this_office_supported)<= {'','0'}
assert not c[c.birth_date_supported_history.eq('')].sex_supported_history.ne('').any()
for r in c[c.birth_date_supported_history.ne('')].to_dict('records'):
 d=f[f.candidate_key.eq(r['candidate_key'])&f.field.eq('birth_date')]
 assert len(d)==1 and d.existing_value.iloc[0]==r['birth_date_supported_history']
 d=f[f.candidate_key.eq(r['candidate_key'])&f.field.eq('sex')]
 assert len(d)==1 and d.existing_value.iloc[0]==r['sex_supported_history']
old=pd.read_csv(ROOT/'data/processed/candidate_fields_reviewed_round4_20261019.csv',dtype=str).fillna('')
old=old.set_index(['candidate_key','field']);new=f.set_index(['candidate_key','field'])
for field in ['incumbency','administrative_experience','posts_Facebook','posts_Instagram','posts_Threads','followers']:
 pd.testing.assert_frame_equal(new.xs(field,level='field'),old.xs(field,level='field'))
assert h.identity_status.str.contains('未填入已核實欄').all()
print('PASS：保存檔hash／列數／鍵；690筆生日性別與欄位表逐人一致；未知首次不填1；未改現任、行政經歷、貼文與追蹤數。')


PASS：保存檔hash／列數／鍵；690筆生日性別與欄位表逐人一致；未知首次不填1；未改現任、行政經歷、貼文與追蹤數。
